# MinIO Delta Lake Access with Spark
This notebook demonstrates how to connect to the MinIO Delta Lake tables using PySpark.

In [2]:
import os
from pyspark.sql import SparkSession

# Set up the SparkSession with Delta Lake and MinIO (S3A) configurations
spark = (
    SparkSession.builder.appName("Jupyter-Delta")
    # Note: 3.5.0 packages for Spark 3.5.0
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # MinIO connectivity
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark session established!")

Spark session established!


In [9]:
! pip install minio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 6.2 MB/s eta 0:00:0000:0100:01


In [10]:
# # List all buckets in MinIO
# fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
# status = fs.listStatus(spark._jvm.org.apache.hadoop.fs.Path("s3a:///"))
# for fileStatus in status:
#     print(fileStatus.getPath())


from minio import Minio

client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin123",
    secure=False,
)

for bucket in client.list_buckets():
    print(bucket.name)

bronze
delta-tables
gold
landing
silver
warehouse


In [7]:
# Example of reading a Delta table (uncomment and replace with your actual bucket/table path)
df = spark.read.format("delta").load("s3a://bronze/stg_reference_branches")
df.show()

# Alternatively, if using Spark SQL:
# spark.sql("SHOW DATABASES").show()

+---------+--------------------+-------------+--------+-----------+---------+
|branch_id|         branch_name|     province|  region|opened_date|is_active|
+---------+--------------------+-------------+--------+-----------+---------+
|   BR0001|    Koshi Branch 001|        Koshi|REGION_2| 2011-02-02|     true|
|   BR0002|  Lumbini Branch 002|      Lumbini|REGION_3| 2012-03-03|     true|
|   BR0003|  Karnali Branch 003|      Karnali|REGION_4| 2013-04-04|     true|
|   BR0004|Sudurpashchim Bra...|Sudurpashchim|REGION_5| 2014-05-05|     true|
|   BR0005|  Karnali Branch 005|      Karnali|REGION_1| 2015-06-06|     true|
|   BR0006|  Lumbini Branch 006|      Lumbini|REGION_2| 2016-07-07|     true|
|   BR0007|  Madhesh Branch 007|      Madhesh|REGION_3| 2017-08-08|     true|
|   BR0008|  Bagmati Branch 008|      Bagmati|REGION_4| 2018-09-09|     true|
|   BR0009|  Karnali Branch 009|      Karnali|REGION_5| 2019-10-10|     true|
|   BR0010|  Bagmati Branch 010|      Bagmati|REGION_1| 2020-11-